## 差分进化算法

### 差分进化算法的起源
差分进化算法（Differential Evolution, DE）是由 Storn 和 Price 于 1995 年提出的一种基于群体智能的随机搜索算法。

起源：为了解决 Chebyshev 多项式拟合问题，Storn 和 Price 提出了一种基于向量差分的进化策略，并于 1996 年在第一届国际进化计算竞赛中展现出优异的性能。

### 差分进化算法与遗传算法的异同
**相同点：**
- 都属于进化算法大类
- 都包含变异、交叉、选择三种基本操作
- 都是基于种群的随机搜索算法

**不同点：**
- DE 的变异操作基于种群中随机个体的差分向量，具有自适应性
- DE 的选择操作采用"贪婪"策略，子代只有优于父代才会被保留
- DE 不需要对变量进行编码/解码，直接在连续空间操作
- DE 的控制参数更少（通常只需种群规模 $NP$、缩放因子 $F$、交叉概率 $CR$）

### 优缺点
**优点：** 结构简单、易于实现、参数少、全局搜索能力强、鲁棒性好
**缺点：** 后期收敛速度可能变慢，对高维问题的搜索效率下降

## 差分进化算法的基本原理

DE 算法的核心思想：利用种群中随机选取的两个个体的差分向量对第三个个体进行扰动，生成变异个体，再通过交叉操作产生试验个体，最后采用贪心选择策略保留较优个体。

### 算法流程
1. **种群初始化**：在解空间中随机生成 $NP$ 个 $D$ 维个体
2. **变异操作**：对每个目标个体 $\mathbf{x}_i$，生成变异个体 $\mathbf{v}_i$
3. **交叉操作**：将目标个体 $\mathbf{x}_i$ 与变异个体 $\mathbf{v}_i$ 进行交叉，生成试验个体 $\mathbf{u}_i$
4. **选择操作**：比较 $\mathbf{x}_i$ 和 $\mathbf{u}_i$ 的适应度，保留较优者
5. 重复步骤 2-4，直到满足终止条件

### 变异策略
常用的变异策略包括：

- **DE/rand/1**：$\mathbf{v}_i = \mathbf{x}_{r1} + F \cdot (\mathbf{x}_{r2} - \mathbf{x}_{r3})$
- **DE/best/1**：$\mathbf{v}_i = \mathbf{x}_{\mathrm{best}} + F \cdot (\mathbf{x}_{r1} - \mathbf{x}_{r2})$
- **DE/rand/2**：$\mathbf{v}_i = \mathbf{x}_{r1} + F \cdot (\mathbf{x}_{r2} - \mathbf{x}_{r3}) + F \cdot (\mathbf{x}_{r4} - \mathbf{x}_{r5})$
- **DE/current-to-best/1**：$\mathbf{v}_i = \mathbf{x}_i + F \cdot (\mathbf{x}_{\mathrm{best}} - \mathbf{x}_i) + F \cdot (\mathbf{x}_{r1} - \mathbf{x}_{r2})$

其中 $F \in [0, 2]$ 为缩放因子，$r1, r2, r3, r4, r5$ 为互不相同且不等于 $i$ 的随机索引。

### 交叉操作
对每个维度 $j$：
$$
u_{i,j} =
\begin{cases}
v_{i,j}, & \text{if } \mathrm{rand}(0,1) \leq CR \text{ or } j = j_{\mathrm{rand}} \\
x_{i,j}, & \text{otherwise}
\end{cases}
$$

其中 $CR \in [0, 1]$ 为交叉概率，$j_{\mathrm{rand}}$ 为随机维数索引，确保至少有一维来自变异个体。

### 选择操作（贪心策略）
以最小化问题为例：
$$
\mathbf{x}_i^{\mathrm{new}} =
\begin{cases}
\mathbf{u}_i, & \text{if } f(\mathbf{u}_i) \leq f(\mathbf{x}_i) \\
\mathbf{x}_i, & \text{otherwise}
\end{cases}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

## 实例计算

本案例使用 **Rastrigin 函数** 作为优化目标，该函数具有大量局部极小值，是测试全局优化算法的经典基准函数：

$$
f(\mathbf{x}) = 10D + \sum_{i=1}^{D} \left[ x_i^2 - 10 \cos(2\pi x_i) \right]
$$

其中 $D$ 为维度，搜索范围通常取 $x_i \in [-5.12, 5.12]$。全局最小值在 $\mathbf{x} = (0, 0, \dots, 0)$ 处，$f_{\min} = 0$。

先画出该函数在二维情况下的图像：

In [ ]:
# 定义 Rastrigin 函数
def rastrigin(x):
    '''Rastrigin 函数，x 形状为 (N, D)'''
    D = x.shape[1]
    return 10 * D + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=1)

# 绘制二维 Rastrigin 函数
x_vals = np.linspace(-5.12, 5.12, 100)
y_vals = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_vals, y_vals)
points = np.column_stack([X.ravel(), Y.ravel()])
Z = rastrigin(points).reshape(X.shape)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax.contour(X, Y, Z, zdir='z', offset=Z.min(), cmap='viridis', alpha=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x, y)')
ax.set_title('Rastrigin 函数图像')
fig.colorbar(surf, shrink=0.5, aspect=10)
plt.show()

### 算法参数设置

In [ ]:
# ==================== DE 参数设置 ====================
NP = 50          # 种群规模
D = 10           # 问题维度
F = 0.5          # 缩放因子
CR = 0.9         # 交叉概率
max_gen = 500    # 最大迭代次数

# 搜索范围
x_min = -5.12
x_max = 5.12

print(f'种群规模 NP = {NP}')
print(f'问题维度 D = {D}')
print(f'缩放因子 F = {F}')
print(f'交叉概率 CR = {CR}')
print(f'最大迭代次数 max_gen = {max_gen}')
print(f'搜索范围: [{x_min}, {x_max}]')

### 种群初始化

In [ ]:
# 初始化种群：在搜索范围内随机生成 NP 个 D 维个体
pop = np.random.uniform(x_min, x_max, (NP, D))

# 计算初始种群的适应度
fitness = rastrigin(pop)

# 记录全局最优
best_idx = np.argmin(fitness)
best_individual = pop[best_idx].copy()
best_fitness = fitness[best_idx]

print(f'初始种群最优适应度: {best_fitness:.6f}')
print(f'初始种群最优个体: {np.round(best_individual, 3)}')

### 变异操作（DE/rand/1）

In [ ]:
def mutation(pop, F):
    '''
    变异操作：DE/rand/1
    对每个目标个体，随机选择三个不同的个体进行差分变异
    '''
    NP, D = pop.shape
    mutant = np.zeros((NP, D))

    for i in range(NP):
        # 随机选择三个互不相同且不等于 i 的索引
        idxs = list(range(NP))
        idxs.remove(i)
        r1, r2, r3 = pop[np.random.choice(idxs, 3, replace=False)]
        # DE/rand/1
        mutant[i] = r1 + F * (r2 - r3)

    return mutant

### 交叉操作（二项式交叉）

In [ ]:
def crossover(pop, mutant, CR):
    '''
    二项式交叉操作
    '''
    NP, D = pop.shape
    trial = np.zeros((NP, D))

    for i in range(NP):
        # 确保至少有一维来自变异个体
        j_rand = np.random.randint(D)
        for j in range(D):
            if np.random.rand() <= CR or j == j_rand:
                trial[i, j] = mutant[i, j]
            else:
                trial[i, j] = pop[i, j]

    return trial

### 选择操作（贪心策略）

In [ ]:
def selection(pop, trial, fitness, func):
    '''
    贪心选择：试验个体优于目标个体时才替换
    '''
    trial_fitness = func(trial)

    # 找出试验个体更优的位置
    improved = trial_fitness < fitness

    # 替换
    pop[improved] = trial[improved]
    fitness[improved] = trial_fitness[improved]

    return pop, fitness

### 边界处理

当变异后的个体超出搜索范围时，采用随机重置策略将其拉回边界内：

In [ ]:
def boundary_repair(pop, x_min, x_max):
    '''
    边界修复：超出范围的维度随机重置到边界内
    '''
    NP, D = pop.shape
    for i in range(NP):
        for j in range(D):
            if pop[i, j] < x_min or pop[i, j] > x_max:
                pop[i, j] = np.random.uniform(x_min, x_max)
    return pop

### DE 主循环

In [ ]:
# 重置种群（确保从头开始）
np.random.seed(42)
pop = np.random.uniform(x_min, x_max, (NP, D))
fitness = rastrigin(pop)
best_idx = np.argmin(fitness)
best_individual = pop[best_idx].copy()
best_fitness = fitness[best_idx]

# 记录迭代过程
fitness_history = [best_fitness]

for gen in range(max_gen):
    # 1. 变异
    mutant = mutation(pop, F)

    # 2. 边界修复
    mutant = boundary_repair(mutant, x_min, x_max)

    # 3. 交叉
    trial = crossover(pop, mutant, CR)

    # 4. 边界修复
    trial = boundary_repair(trial, x_min, x_max)

    # 5. 选择
    pop, fitness = selection(pop, trial, fitness, rastrigin)

    # 6. 更新全局最优
    gen_best_idx = np.argmin(fitness)
    if fitness[gen_best_idx] < best_fitness:
        best_fitness = fitness[gen_best_idx]
        best_individual = pop[gen_best_idx].copy()

    fitness_history.append(best_fitness)

    # 打印进度
    if (gen + 1) % 50 == 0:
        print(f'Generation {gen + 1:4d}, Best Fitness: {best_fitness:.8f}')

print()
print(f'优化完成！')
print(f'全局最优适应度: {best_fitness:.10f}')
print(f'全局最优解: {np.round(best_individual, 4)}')

### 收敛曲线

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(fitness_history, linewidth=2, color='#2c7bb6')
plt.xlabel('迭代次数', fontsize=12)
plt.ylabel('最优适应度', fontsize=12)
plt.title('DE 算法收敛曲线 (Rastrigin 函数, D=10)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

### 二维搜索过程可视化

为了直观展示 DE 的搜索过程，我们在二维 Rastrigin 函数上运行算法，并绘制种群分布：

In [ ]:
# 在二维问题上运行 DE
np.random.seed(42)
NP_2d = 30
D_2d = 2
F_2d = 0.5
CR_2d = 0.9
max_gen_2d = 200

pop_2d = np.random.uniform(x_min, x_max, (NP_2d, D_2d))
fitness_2d = rastrigin(pop_2d)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# 等高线背景
X_bg, Y_bg = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(x_min, x_max, 200))
Z_bg = rastrigin(np.column_stack([X_bg.ravel(), Y_bg.ravel()])).reshape(X_bg.shape)

snapshots = [0, 1, 3, 10, 50, max_gen_2d]
for ax_idx, snap in enumerate(snapshots):
    if snap > 0:
        for _ in range(snap if ax_idx == 0 else snap - snapshots[ax_idx - 1]):
            mutant_2d = mutation(pop_2d, F_2d)
            mutant_2d = boundary_repair(mutant_2d, x_min, x_max)
            trial_2d = crossover(pop_2d, mutant_2d, CR_2d)
            trial_2d = boundary_repair(trial_2d, x_min, x_max)
            pop_2d, fitness_2d = selection(pop_2d, trial_2d, fitness_2d, rastrigin)

    ax = axes[ax_idx]
    ax.contourf(X_bg, Y_bg, Z_bg, levels=30, cmap='viridis', alpha=0.6)
    ax.scatter(pop_2d[:, 0], pop_2d[:, 1], c='red', s=10, alpha=0.7,
               edgecolors='white', linewidth=0.5)
    best_idx_2d = np.argmin(fitness_2d)
    ax.scatter(pop_2d[best_idx_2d, 0], pop_2d[best_idx_2d, 1],
               c='yellow', s=100, marker='*', edgecolors='black', linewidth=1)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(x_min, x_max)
    ax.set_title(f'第 {snap} 代')

plt.suptitle('DE 算法搜索过程（二维 Rastrigin 函数）', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 一点讨论：参数敏感性分析

缩放因子 $F$ 和交叉概率 $CR$ 是 DE 算法的两个关键参数：

- **$F$（缩放因子）**：控制差分向量的缩放幅度。较大的 $F$ 有利于全局搜索，但可能降低收敛精度；较小的 $F$ 有利于局部搜索，但可能陷入局部最优。通常建议 $F \in [0.4, 0.9]$
- **$CR$（交叉概率）**：控制变异个体对试验个体的贡献程度。较大的 $CR$ 加速收敛；较小的 $CR$ 有利于保持种群多样性。通常建议 $CR \in [0.1, 0.9]$

### 不同变异策略的特点
- **DE/rand/1**：最经典的策略，探索能力强，收敛较慢
- **DE/best/1**：利用当前最优个体引导搜索，收敛快但容易早熟
- **DE/rand/2**：使用两个差分向量，扰动更大，探索能力更强
- **DE/current-to-best/1**：平衡了探索和开发，是实际应用中最常用的策略之一